# Prompt Template （提示词模版）

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key="your-api-key-here",# type: ignore
    temperature=0.9
    
)

prompt = PromptTemplate.from_template(
    "请介绍{city}的旅游景点"
)

text = prompt.format(
    city="上海"
)

result = llm.invoke(text)

print(result.content)

### 1.创建Chat Prompt Template 

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key="your-api-key-here",# type: ignore
    temperature=0.9   
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","你是一位资深Python老师"),
        ("human","解释一下{topic}")
    ]
)

messages = prompt.invoke(
    {
        "topic":"装饰器"
    }
)

result = llm.invoke(messages)

print(result.content)

In [ ]:
from datetime import datetime

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough
)

# =====================
# 1. 创建大模型
# =====================

llm = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key="your-api-key-here",# type: ignore
    temperature=0.9,
    max_completion_tokens=100 # type: ignore
)

# =====================
# 2. 数据预处理函数
# =====================

def preprocess(topic: str):

    print("原始输入:", topic)

    return {
        "topic": topic.upper()
    }

# Runnable包装
preprocess_node = RunnableLambda(preprocess)

# =====================
# 3. 添加额外字段
# =====================

add_time = RunnablePassthrough.assign(
    current_time=lambda _: datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)

# =====================
# 4. Prompt模板
# =====================

prompt = ChatPromptTemplate.from_template(
    """
你是一位资深Python老师。

当前时间：
{current_time}

请详细讲解：

{topic}
"""
)

# =====================
# 5. 输出解析器
# =====================

parser = StrOutputParser()

# =====================
# 6. 构建链
# =====================

chain = (
    preprocess_node
    | add_time
    | prompt
    | llm
    | parser
)

# =====================
# 7. 执行
# =====================

result = chain.invoke(
    "python装饰器"
)

print("\n最终结果：")
print(result)